# NumPy Random Numbers

> 📘 **Python Mastery** · Module 10 — NumPy · Lesson 7/7

Randomness is the spice of machine learning: weight initialization, shuffling data, splitting train/test sets, simulating noise. This lesson teaches the modern Generator API — and why a fixed seed is the difference between an experiment and an anecdote.

## 🎯 Learning Objectives

- Create random streams the modern way with `np.random.default_rng`
- Draw integers, uniform floats and normal (Gaussian) samples
- Sample populations with `choice`, including weighted probabilities
- Shuffle arrays correctly: `shuffle` (in place) vs `permutation` (copy)
- Read legacy `np.random.*` code you meet online
- Reproduce results by seeding, and sanity-check simulated distributions

## 1. The Modern API: np.random.default_rng

Since NumPy 1.17 the recommended style creates a **Generator** object that owns its own private random state. You draw values by calling methods on it.

**Syntax:**
```python
rng = np.random.default_rng(seed=42)   # seed optional but recommended
rng.random()                           # then call METHODS on rng
```

In [ ]:
import numpy as np

rng = np.random.default_rng(seed=42)   # a Generator with private state

print(type(rng))
print("one random float :", rng.random())
print("another one      :", rng.random())   # each draw advances the stream

## 2. Random Integers and Uniform Floats

`rng.integers(low, high)` draws whole numbers — high is EXCLUSIVE unless you pass `endpoint=True`. `rng.random()` gives floats in `[0, 1)`; `rng.uniform(a, b)` spreads them over any interval.

**Syntax:**
```python
rng.integers(low, high, size=n)              # low .. high-1
rng.integers(low, high, size=n, endpoint=True)
rng.random(size=n)                           # floats in [0, 1)
rng.uniform(low, high, size=n)               # floats in [low, high)
```

In [ ]:
import numpy as np

rng = np.random.default_rng(seed=7)

print("single die roll    :", rng.integers(1, 7))            # 1..6, 7 excluded
print("five dice rolls    :", rng.integers(1, 7, size=5))
print("inclusive endpoint :", rng.integers(1, 6, size=5, endpoint=True))   # 1..6 too
print("4-digit PIN        :", "".join(str(d) for d in rng.integers(0, 10, size=4)))

In [ ]:
import numpy as np

rng = np.random.default_rng(seed=7)

print("uniform [0, 1)  :", np.round(rng.random(size=3), 3))
print("uniform [10,20) :", np.round(rng.uniform(10, 20, size=3), 3))

## 3. Normal (Gaussian) Values

Nature clusters around averages — heights, measurement errors, exam results. `rng.normal(mean, std, size)` draws from the bell curve; it is THE distribution of synthetic ML data.

**Syntax:**
```python
rng.normal(loc=mean, scale=std, size=n)
rng.standard_normal(size=n)     # shortcut: mean 0, std 1
```

In [ ]:
import numpy as np

rng = np.random.default_rng(seed=42)

heights = rng.normal(loc=170, scale=8, size=5)   # mean 170 cm, sd 8 cm
print("sample heights:", np.round(heights, 1))

errors = rng.standard_normal(size=4)             # zero-centered noise
print("standard noise :", np.round(errors, 3))

## 4. choice: Pick From Any Population

`rng.choice` draws elements from an arbitrary array — with or without replacement, and optionally weighted so some items are picked more often.

**Syntax:**
```python
rng.choice(population, size=k)                     # with replacement
rng.choice(population, size=k, replace=False)      # unique picks
rng.choice(population, size=k, p=probabilities)    # weighted sampling
```

The probabilities in `p=` must sum to 1.

In [ ]:
import numpy as np

rng = np.random.default_rng(seed=3)

students = np.array(["Rafi", "Nadia", "Omar", "Priya"])

print("pick 2, no repeats  :", rng.choice(students, size=2, replace=False))
print("pick 5, repeats ok  :", rng.choice(students, size=5))

# Weighted: Nadia volunteers half the time
picks = rng.choice(students, size=20, p=[0.2, 0.5, 0.2, 0.1])
names, counts = np.unique(picks, return_counts=True)
for name, n in zip(names, counts):
    print(name, "was picked", int(n), "times")

## 5. shuffle vs permutation

Both reorder randomly, but they differ in place: `rng.shuffle(arr)` mixes the array ITSELF (returns None!), while `rng.permutation(arr)` returns a shuffled COPY and leaves the original intact.

**Syntax:**
```python
rng.shuffle(deck)          # in place - deck is changed, returns None
mixed = rng.permutation(fresh)   # fresh untouched
rng.permutation(10)        # shortcut: shuffled range 0..9
```

In [ ]:
import numpy as np

rng = np.random.default_rng(seed=11)

deck = np.arange(1, 11)          # cards numbered 1..10
result = rng.shuffle(deck)       # shuffles IN PLACE...
print("shuffle returned:", result)      # ...and returns None!
print("deck now        :", deck)

fresh = np.arange(1, 11)
mixed = rng.permutation(fresh)   # returns a shuffled COPY
print("permutation     :", mixed)
print("original intact :", fresh)

## 6. The Legacy API: np.random.* Everywhere Online

Before NumPy 1.17, random functions lived directly on `np.random` and shared ONE hidden global generator. Millions of tutorials still use that style, so read it fluently — just write new code with `default_rng`.

| Modern (recommended) | Legacy (common online) | Purpose |
|---|---|---|
| `np.random.default_rng(seed)` | `np.random.seed(seed)` | create / control randomness |
| `rng.random(n)` | `np.random.rand(n)` or `.random(n)` | uniform floats in [0, 1) |
| `rng.integers(low, high, size)` | `np.random.randint(low, high, size)` | random integers |
| `rng.normal(mu, sigma, size)` | `np.random.randn(size)` / `.normal(...)` | Gaussian samples |
| `rng.choice(a, size, p=p)` | `np.random.choice(a, size, p=p)` | sample a population |
| `rng.shuffle(x)` | `np.random.shuffle(x)` | shuffle in place |
| `rng.permutation(x)` | `np.random.permutation(x)` | shuffled copy |

Why modern wins: every Generator carries its OWN state. Legacy calls share global state, so any library calling `np.random.seed(...)` silently rewrites YOUR randomness too.

In [ ]:
import numpy as np

# Legacy style - what older tutorials show
np.random.seed(42)
print("legacy randint:", np.random.randint(0, 10, size=5))
print("legacy randn  :", np.round(np.random.randn(3), 3))

# Modern equivalent - state lives inside the rng object, not in globals
rng = np.random.default_rng(seed=42)
print("modern integers:", rng.integers(0, 10, size=5))

## 7. Reproducibility: Seed Your Experiments

Same seed, same numbers — forever. Seeding is how science stays reviewable: a teammate (or future-you) can rerun your notebook and get byte-identical splits, weights and results. In Module 13 this becomes ritual: fix ONE seed at the top of every ML experiment.

**Syntax:**
```python
rng = np.random.default_rng(SEED)   # one seed -> identical stream every run
```

In [ ]:
import numpy as np

first = np.random.default_rng(seed=123).integers(0, 100, size=5)
second = np.random.default_rng(seed=123).integers(0, 100, size=5)

print("first run :", first)
print("second run:", second)
print("identical?", np.array_equal(first, second))

In [ ]:
import numpy as np

SEED = 42                          # typical ML experiment pattern:
rng = np.random.default_rng(SEED)  # ONE generator created once, used everywhere

weights = rng.normal(0, 0.01, size=4)     # model weight initialization
noise = rng.normal(0, 0.1, size=4)        # synthetic measurement noise
split = rng.permutation(10)[:3]           # indices for a tiny train split

print("weights   :", np.round(weights, 4))
print("noise     :", np.round(noise, 4))
print("split idx :", split)
# Re-run this cell: identical output every time. That is reproducible science.

## 8. Sanity Check: Does normal() Behave Itself?

Trust, but verify. Draw 100,000 Gaussian samples and confirm the empirical mean/std land near the requested parameters, plus the famous 68–95 rule of bell curves.

**Syntax:**
```python
sample = rng.normal(mu, sigma, size=100_000)
sample.mean(), sample.std()
```

In [ ]:
import numpy as np

rng = np.random.default_rng(seed=0)

sample = rng.normal(loc=100, scale=15, size=100_000)   # IQ-like distribution

print("samples      :", sample.size)
print("empirical mean:", round(sample.mean(), 2), "(target 100)")
print("empirical std :", round(sample.std(), 2), "(target 15)")
print("within 1 sd:", round(np.mean(np.abs(sample - 100) < 15) * 100, 1), "% (theory ~68%)")
print("within 2 sd:", round(np.mean(np.abs(sample - 100) < 30) * 100, 1), "% (theory ~95%)")

## 9. Mini Project: Simulating Dice

Everything from this module in two cells: generate dice rolls, count outcomes with `np.unique`, and let the law of large numbers draw the histogram for us — no matplotlib needed yet.

In [ ]:
import numpy as np

rng = np.random.default_rng(seed=42)

rolls = rng.integers(1, 7, size=1_000)   # 1000 fair six-sided dice rolls
faces, counts = np.unique(rolls, return_counts=True)

for face, count in zip(faces, counts):
    bar = "#" * (count // 5)
    print(f"face {face}: {count:4d}  {bar}")

In [ ]:
import numpy as np

rng = np.random.default_rng(seed=1)

die1 = rng.integers(1, 7, size=10_000)
die2 = rng.integers(1, 7, size=10_000)
totals = die1 + die2                      # the classic two-dice game

faces, counts = np.unique(totals, return_counts=True)
for face, count in zip(faces, counts):
    print(f"total {face:2d}: {count / totals.size:.1%}")

best = faces[counts.argmax()]
print("most likely total:", best, "- exactly what probability theory predicts")

> 🔍 **Under the Hood:** Computers cannot flip real coins. `default_rng` hands you a Generator wrapping the PCG64 pseudo-random number generator: a deterministic formula that stretches one seed into an apparently endless stream of statistically random-looking numbers. Same seed → same stream → reproducibility. The legacy `np.random.*` functions drive a single global Mersenne Twister whose state hides in module globals — which is why any library calling `np.random.seed()` could silently break your experiment. Generators keep state ON the object: isolation by construction. And remember: none of this is cryptographic — use Python's `secrets` module for anything security-related.

## ⚠️ Common Mistakes & Gotchas

| Mistake | Problem | Fix |
|---|---|---|
| Calling `np.random.seed(...)` mid-project | Resets the GLOBAL legacy state under everyone's feet | Use `default_rng(seed)` instances instead |
| Forgetting `high` is exclusive in `rng.integers(1, 7)` | Die rolls never show 7... but also break if you wrote `(1, 6)` expecting 6 included | Pass `endpoint=True` or think `[low, high)` |
| Assigning `shuffled = rng.shuffle(x)` | `shuffle` returns None — shuffled is now None | Shuffle in place, or use `rng.permutation(x)` |
| Creating `default_rng(42)` INSIDE a loop | Every iteration restarts the same stream — fake variety | Create the generator once, outside the loop |
| Using NumPy randomness for passwords/tokens | PCG64 is predictable from its outputs | Use the `secrets` module for security needs |

## 💡 Best Practices & Pro Tips

- Define ONE seed constant at the top of a notebook and derive everything from a single `rng`.
- Pass the `rng` object INTO functions rather than drawing from global state — testable and reproducible.
- Draw whole arrays at once with `size=(rows, cols)` instead of looping single draws; it is faster and cleaner.
- After simulating, always sanity-check with aggregations from Lesson 5 (`mean`, `std`, `unique`).
- **AI-engineering relevance:** weight init, dropout masks, dataset shuffling and synthetic augmentation are all `default_rng` calls. Published ML papers list their seeds precisely because of this lesson — it foreshadows the experiment protocol in Module 13.

## 📌 Summary

| Tool | What it does | Example |
|---|---|---|
| `np.random.default_rng(seed)` | Modern Generator with private state | `rng = default_rng(42)` |
| `rng.integers(lo, hi, size)` | Random ints, hi exclusive (`endpoint=True` to include) | `rng.integers(1, 7, size=5)` |
| `rng.random(size)` | Uniform floats in [0, 1) | `rng.random(3)` |
| `rng.uniform(a, b, size)` | Uniform floats in [a, b) | `rng.uniform(10, 20, 3)` |
| `rng.normal(mu, sigma, size)` | Gaussian samples | `rng.normal(170, 8, 5)` |
| `rng.choice(a, size, p=p)` | Sample a population, optionally weighted | `rng.choice(names, p=[.5,.5])` |
| `rng.shuffle(x)` | Shuffle IN PLACE (returns None) | `rng.shuffle(deck)` |
| `rng.permutation(x)` | Shuffled copy | `mixed = rng.permutation(x)` |
| legacy `np.random.rand/randint/randn/seed` | Old global API, still everywhere online | read-only fluency |

Key takeaways:
- New code uses `default_rng`; old code uses the global `np.random.*` namespace — recognize both.
- A seed turns luck into science: same seed → identical stream.
- `shuffle` mutates and returns None; `permutation` copies.
- Verify simulations statistically — empirical mean/std should match theory within ~0.5%.

## 🔗 Next Lesson

- **Module 10 complete!** Continue to **Module 11 — Pandas** ([11_Pandas](../../11_Pandas/README.md)), where these array skills become DataFrames — labeled tables built directly on top of NumPy.